# MVO4 portfolio research
Research workflow covering preselection, allocation objectives and benchmark evaluation. Requires separately supplied workbooks. No historical performance claim is made by this package.


### Experiment cell 1


In [ ]:
import sklearn
sklearn.__version__

### Experiment cell 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from pathlib import Path

### Experiment cell 3


In [ ]:
import yfinance as yf

# Compatibility wrapper: replaces pdr.get_data_yahoo with yfinance.download
class pdr:
    @staticmethod
    def get_data_yahoo(tickers, start=None, end=None, **kwargs):
        data = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False, threads=False)
        return data

### Experiment cell 4


In [ ]:
def modify_date(date):
    return date[:10]

### Experiment cell 5


In [ ]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

### Experiment cell 6


In [ ]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

### Experiment cell 7


In [ ]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

### Experiment cell 8


In [ ]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

### Experiment cell 9


In [ ]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=np.nan
    dfMVOweight['adjusted proportion']=np.nan
    dfMVOweight['direct']=np.nan
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight.iloc[0]:
            dfMVOweight.iloc[i,5]=allocationweight.iloc[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight.iloc[1])):
        dfMVOweight.iloc[i,6]=allocationweight.iloc[0]
    for i in range(int(allocationweight.iloc[1]),int(allocationweight.iloc[1]+allocationweight.iloc[3])):
        dfMVOweight.iloc[i,6]=allocationweight.iloc[2]
    return dfMVOweight

### Experiment cell 10


In [ ]:
def output(weight):
    if stock_type == "S&P100":
      OEX=yf.download("^OEX", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]

    elif stock_type == "S&P500":
      OEX=yf.download("SPY", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]

    if isinstance(OEX, pd.DataFrame):
        OEX = OEX.iloc[:, 0]
    stock_price=yf.download(list(weight.index), start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX.iloc[0])
    injected_result["oex"]=S2*OEX
    return injected_result

### Experiment cell 11


In [ ]:
# Preselection parameters set up
import os
DATA_DIR = Path(os.environ.get("MVO_DATA_DIR", "data/private"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

### Experiment cell 12


In [ ]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel(DATA_DIR / "All_SwanBeta100_MVO2.xlsx", index_col=0)
else:
    portfolio_beta = pd.read_excel(DATA_DIR / "All_SwanBeta500.xlsx", index_col=0)

### Experiment cell 13


In [ ]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

### Experiment cell 14


In [ ]:
portfolio_beta

### Experiment cell 15


In [ ]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

### Experiment cell 16


In [ ]:
selected_corr

### Experiment cell 17


In [ ]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

### Experiment cell 18


In [ ]:
portfolio_beta

### Experiment cell 19


In [ ]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"].iloc[0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"].iloc[first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,12):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"].iloc[0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"].iloc[first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=2:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

### Experiment cell 20


In [ ]:
ct

### Experiment cell 21


In [ ]:
res

### Experiment cell 22


In [ ]:
preselected_stock.info()

### Experiment cell 23


In [ ]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

### Experiment cell 24


In [ ]:
order = list(preselected_stock.index)
random.shuffle(order)
preselected_stock = preselected_stock.reindex(index = order)
preselected_stock

### Experiment cell 25


In [ ]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

### Experiment cell 26


In [ ]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

### Experiment cell 27


In [ ]:
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Close"]

### Experiment cell 28


In [ ]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

### Experiment cell 29


In [ ]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel(DATA_DIR / "MarketDD.xlsx", index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

### Experiment cell 30


In [ ]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv(DATA_DIR / "trainingMarketdrop.csv")

### Experiment cell 31


In [ ]:
market_drop.to_csv("market_drop(op).csv")

### Experiment cell 32


In [ ]:
market_drop

### Experiment cell 33


In [ ]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

### Experiment cell 34


In [ ]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

### Experiment cell 35


In [ ]:
yahooDataFrame.to_csv('yahooDataFrame(op).csv')
yahooDataFrame

### Experiment cell 36


In [ ]:
#Set the paramter for MVO process
Portfolio_Date=end

### Experiment cell 37


In [ ]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

### Experiment cell 38


In [ ]:
# Risk-free rate used downstream by the optimization.
# The legacy Quandl call and embedded API key were removed.
rf_week = pd.Series([0.01])  # 1% annual risk-free rate

### Experiment cell 39


In [ ]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

### Experiment cell 40


In [ ]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

### Experiment cell 41


In [ ]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

### Experiment cell 42


In [ ]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

### Experiment cell 43


In [ ]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return, axis=0)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return, axis=0)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return, axis=0)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return, axis=0)*np.sqrt(24);

### Experiment cell 44


In [ ]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

### Experiment cell 45


In [ ]:
optdata = pd.read_excel(DATA_DIR / "optdata_MVO4.xlsx", index_col=0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index].copy()

# CHANGE 3: Adjust P_fwd for stock splits that happened AFTER the optdata date (2020-02-15).
# optdata has pre-split price targets; Yahoo Close is retroactively split-adjusted.
optdata_date = pd.Timestamp("2020-02-15")
for stock in P_fwd.index:
    try:
        ticker_obj = yf.Ticker(stock)
        splits = ticker_obj.splits
        if splits is not None and len(splits) > 0:
            if splits.index.tz is not None:
                cutoff_tz = optdata_date.tz_localize(splits.index.tz)
            else:
                cutoff_tz = optdata_date
            post_splits = splits[splits.index > cutoff_tz]
            if len(post_splits) > 0:
                cumulative_ratio = post_splits.prod()
                P_fwd[stock] = P_fwd[stock] / cumulative_ratio
                print(f"  {stock}: adjusted P_fwd by {cumulative_ratio:.0f}:1 split -> P_fwd={P_fwd[stock]:.2f}")
    except Exception as e:
        print(f"  {stock}: split check error: {e}")

R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(0.0, index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0
# CHANGE 1: Use R_fwd (not Rfwd_adjusted) — floor negative values at 0
R_fwd[R_fwd<0]=0

### Experiment cell 46


In [ ]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
# CHANGE 1: Use R_fwd instead of Rfwd_adjusted for SR and PR_SR
preselected_stock["Rfwd_adjusted"]=R_fwd.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info(op).csv")
p=FFC.MA_now
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank(op).csv")


### Experiment cell 47


In [ ]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
# CHANGE 1: Use R_fwd instead of Rfwd_adjusted
E_2 = R_fwd/fall
E_4 = R_fwd
std =  np.std(np.exp(one_year_weekly_return)-1, axis=0)*np.sqrt(52)

### Experiment cell 48


In [ ]:
# ============================================================
# COMPUTE PER-STOCK BETA FOR PR OPTIMIZATION
# ============================================================
from scipy.optimize import minimize

# NOTE: Do NOT modify preselected_stock - it feeds post_allocation downstream.
# E_beta is used ONLY by the optimizer cells below.

# Use actual per-stock swan betas (Swan4..Swan8) for optimizer
swan_cols = [c for c in preselected_stock.columns if c.startswith('Swan')]
swan_beta_matrix = preselected_stock.loc[selected_stock_list, swan_cols].copy()

# Per-stock expected beta = simple mean across available swan periods
E_beta = swan_beta_matrix.mean(axis=1)
E_beta = E_beta.fillna(E_beta.mean())
E_PR = 1 - E_beta

# Fill NaN for covariance computation
for col in swan_cols:
    swan_beta_matrix[col] = swan_beta_matrix[col].fillna(E_beta)

beta_cov_raw = np.cov(swan_beta_matrix.values, rowvar=True, bias=False)
beta_cov_matrix = pd.DataFrame(
    beta_cov_raw + np.eye(len(selected_stock_list)) * 1e-6,
    index=selected_stock_list, columns=selected_stock_list
)

print(f"E_beta range: [{E_beta.min():.4f}, {E_beta.max():.4f}]")
print(f"E_PR range:   [{E_PR.min():.4f}, {E_PR.max():.4f}]")
print(f"Beta spread:  {E_beta.max()-E_beta.min():.4f}")
print(f"\nLowest beta (best PR):")
print(E_beta.sort_values().head())

### Experiment cell 49


In [ ]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

### Experiment cell 50


In [ ]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()

### Experiment cell 51


In [ ]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

### Experiment cell 52


In [ ]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1

### Experiment cell 53


In [ ]:
# ============================================================
# BASELINE SR OPTIMIZATION: Portfolio Beta & PR Metrics
# ============================================================
beta_orig = weight1 @ E_beta.values
pr_orig = 1 - beta_orig
print(f"Baseline SR Optimization:")
print(f"  Portfolio Beta:      {beta_orig:.4f}")
print(f"  Protection Ratio:    {pr_orig:.4f}")
print(f"  Sharpe Ratio:        {sharpe_ratio1:.4f}")
print(f"  Expected Return:     {R1:.4f}")
print(f"  Volatility:          {sd1:.4f}")

### Experiment cell 54


In [ ]:
# ============================================================
# OPTIMIZATION 1: Maximize (1-Beta) / SD(Beta)
# ============================================================
print("="*60)
print("OPTIMIZATION 1: Maximize (1-Beta) / SD(Beta)")
print("="*60)

n = selected_stock_number
bounds_opt = [(0, w1)] * n
constraints_opt = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
w0_opt = np.ones(n) / n

E_beta_vals = E_beta.values
beta_cov_vals = beta_cov_matrix.values
cov_vals = cov.values
E_2_vals = (E_2 / 100).values

def neg_pr_over_sd_beta(w):
    """Maximize (1-Beta) / SD(Beta)"""
    port_beta = w @ E_beta_vals
    port_pr = 1 - port_beta
    beta_var = w @ beta_cov_vals @ w
    beta_sd = np.sqrt(max(beta_var, 1e-12))
    return -port_pr / beta_sd

res_m1 = minimize(neg_pr_over_sd_beta, w0_opt, method='SLSQP',
                  bounds=bounds_opt, constraints=constraints_opt)
weight_m1 = res_m1.x.copy()
weight_m1[weight_m1 < 1e-4] = 0
weight_m1 = weight_m1 / weight_m1.sum()

beta_m1 = weight_m1 @ E_beta_vals
pr_m1 = 1 - beta_m1
sd_beta_m1 = np.sqrt(weight_m1 @ beta_cov_vals @ weight_m1)
sd_ret_m1 = np.sqrt(weight_m1 @ cov_vals @ weight_m1)
ret_m1 = weight_m1 @ E_2_vals
sr_m1 = (ret_m1 - 0.01) / sd_ret_m1 if sd_ret_m1 > 0 else 0
obj1_val = pr_m1 / sd_beta_m1

wdf_m1 = pd.DataFrame(weight_m1, index=selected_stock_list, columns=['weight'])
wdf_m1_clean = wdf_m1[wdf_m1.weight > 0.001]
print(wdf_m1_clean.sort_values('weight', ascending=False))
print(f"\nPortfolio Beta: {beta_m1:.4f}")
print(f"PR (1-Beta):    {pr_m1:.4f}")
print(f"SD(Beta):       {sd_beta_m1:.4f}")
print(f"Objective:      {obj1_val:.4f}")
print(f"Sharpe Ratio:   {sr_m1:.4f}")
print(f"Exp Return:     {ret_m1:.4f}")
print(f"Volatility:     {sd_ret_m1:.4f}")
print(f"Optimizer success: {res_m1.success}")

### Experiment cell 55


In [ ]:
# ============================================================
# OPTIMIZATION 2: Maximize (1-Beta) / SD(1-Beta)
# Note: SD(1-Beta) = SD(Beta) since Var(1-X) = Var(X)
# Mathematically identical to Method 1, included for completeness.
# ============================================================
print("="*60)
print("OPTIMIZATION 2: Maximize (1-Beta) / SD(1-Beta)")
print("="*60)
print("Note: SD(1-B) = SD(B) since Var(1-X)=Var(X), same as Method 1.\n")

def neg_pr_over_sd_pr(w):
    """Maximize (1-Beta) / SD(1-Beta)"""
    port_beta = w @ E_beta_vals
    port_pr = 1 - port_beta
    pr_var = w @ beta_cov_vals @ w  # SD(1-B) = SD(B)
    pr_sd = np.sqrt(max(pr_var, 1e-12))
    return -port_pr / pr_sd

res_m2 = minimize(neg_pr_over_sd_pr, w0_opt, method='SLSQP',
                  bounds=bounds_opt, constraints=constraints_opt)
weight_m2 = res_m2.x.copy()
weight_m2[weight_m2 < 1e-4] = 0
weight_m2 = weight_m2 / weight_m2.sum()

beta_m2 = weight_m2 @ E_beta_vals
pr_m2 = 1 - beta_m2
sd_pr_m2 = np.sqrt(weight_m2 @ beta_cov_vals @ weight_m2)
sd_ret_m2 = np.sqrt(weight_m2 @ cov_vals @ weight_m2)
ret_m2 = weight_m2 @ E_2_vals
sr_m2 = (ret_m2 - 0.01) / sd_ret_m2 if sd_ret_m2 > 0 else 0
obj2_val = pr_m2 / sd_pr_m2

wdf_m2 = pd.DataFrame(weight_m2, index=selected_stock_list, columns=['weight'])
wdf_m2_clean = wdf_m2[wdf_m2.weight > 0.001]
print(wdf_m2_clean.sort_values('weight', ascending=False))
print(f"\nPortfolio Beta: {beta_m2:.4f}")
print(f"PR (1-Beta):    {pr_m2:.4f}")
print(f"SD(1-Beta):     {sd_pr_m2:.4f}")
print(f"Objective:      {obj2_val:.4f}")
print(f"Sharpe Ratio:   {sr_m2:.4f}")
print(f"Optimizer success: {res_m2.success}")

### Experiment cell 56


In [ ]:
# ============================================================
# OPTIMIZATION 3: Maximize SR * (1-Beta) / SD(1-Beta)
# ============================================================
print("="*60)
print("OPTIMIZATION 3: Maximize SR * (1-Beta) / SD(1-Beta)")
print("="*60)

def neg_sr_times_pr_over_sd_pr(w):
    """Maximize SR * (1-Beta) / SD(1-Beta)"""
    # Sharpe Ratio component
    port_ret = w @ E_2_vals
    ret_var = w @ cov_vals @ w
    ret_sd = np.sqrt(max(ret_var, 1e-12))
    sr = (port_ret - 0.01) / ret_sd

    # PR / SD(1-B) component
    port_beta = w @ E_beta_vals
    port_pr = 1 - port_beta
    pr_var = w @ beta_cov_vals @ w
    pr_sd = np.sqrt(max(pr_var, 1e-12))

    return -(sr * port_pr / pr_sd)

res_m3 = minimize(neg_sr_times_pr_over_sd_pr, w0_opt, method='SLSQP',
                  bounds=bounds_opt, constraints=constraints_opt)
weight_m3 = res_m3.x.copy()
weight_m3[weight_m3 < 1e-4] = 0
weight_m3 = weight_m3 / weight_m3.sum()

beta_m3 = weight_m3 @ E_beta_vals
pr_m3 = 1 - beta_m3
sd_pr_m3 = np.sqrt(weight_m3 @ beta_cov_vals @ weight_m3)
sd_ret_m3 = np.sqrt(weight_m3 @ cov_vals @ weight_m3)
ret_m3 = weight_m3 @ E_2_vals
sr_m3 = (ret_m3 - 0.01) / sd_ret_m3 if sd_ret_m3 > 0 else 0
obj3_val = sr_m3 * pr_m3 / sd_pr_m3

wdf_m3 = pd.DataFrame(weight_m3, index=selected_stock_list, columns=['weight'])
wdf_m3_clean = wdf_m3[wdf_m3.weight > 0.001]
print(wdf_m3_clean.sort_values('weight', ascending=False))
print(f"\nPortfolio Beta: {beta_m3:.4f}")
print(f"PR (1-Beta):    {pr_m3:.4f}")
print(f"SD(1-Beta):     {sd_pr_m3:.4f}")
print(f"Sharpe Ratio:   {sr_m3:.4f}")
print(f"Objective:      {obj3_val:.4f}")
print(f"Exp Return:     {ret_m3:.4f}")
print(f"Volatility:     {sd_ret_m3:.4f}")
print(f"Optimizer success: {res_m3.success}")
# Save Method 3 weights for the backtest section below.
wdf_m3.to_csv(DATA_DIR / 'weights_method3_SR_PR_over_SDPR.csv')

### Experiment cell 57


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
from copy import deepcopy
import yfinance as yf

### Experiment cell 58


In [ ]:
# PARAMETERS
method_number = 3
method_name = "M3: SR*PR/SD(1-Beta)"
start_swan_date = dt(2020, 2, 15)
end_swan_date = dt(2021, 3, 23)
stock_type = "S&P100"
N = 7
MA_days = 30

### Experiment cell 59


In [ ]:
# LOAD WEIGHTS
weight = pd.read_csv(DATA_DIR / "weights_method3_SR_PR_over_SDPR.csv", index_col=0)
weight = weight[weight['weight'] >= 0.030]
weight['weight'] = weight['weight'] / weight['weight'].sum()
print(f"Method {method_number}: {method_name}")
print(f"{len(weight)} stocks after filtering (>=3% weight)")
print(weight)

### Experiment cell 60


In [ ]:
# LOAD PRESELECTED STOCK INFO
preselected_stock = pd.read_csv(DATA_DIR / "preselected_Stocks_info(op).csv", index_col=0)

### Experiment cell 61


In [ ]:
# DOWNLOAD PRICES — Close is already split-adjusted in yfinance
def get_data_yahoo(tickers, start=None, end=None):
    data = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False, threads=False)
    return data

if stock_type == "S&P100":
    OEX = get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Close"]
elif stock_type == "S&P500":
    OEX = get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Close"]

if isinstance(OEX, pd.DataFrame):
    OEX = OEX.iloc[:, 0]

stock_price = get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Close"]
stock_price = stock_price[weight.index]

stock_price.to_csv(f"tradeinfo_m{method_number}.csv")
OEX.to_csv(f"oex_m{method_number}.csv")

print(f"\nStock price shape: {stock_price.shape}")
print(f"OEX shape: {OEX.shape}")

### Experiment cell 62


In [ ]:
# CALCULATE ACTUAL PR (Swan 9: Feb 19 - Mar 23, 2020)
start_prices = stock_price.loc["2020-02-19"]
end_prices = stock_price.loc["2020-03-23"]
stock_drop = abs(end_prices / start_prices - 1)

OEX_start = OEX.loc["2020-02-19"]
OEX_end = OEX.loc["2020-03-23"]
if hasattr(OEX_start, 'iloc'):
    OEX_start = OEX_start.iloc[0]
if hasattr(OEX_end, 'iloc'):
    OEX_end = OEX_end.iloc[0]
market_drop_val = float(abs(OEX_end / OEX_start - 1))

beta = stock_drop / market_drop_val
actual_PR = 1 - float(beta.T @ weight['weight'])
print(f"\nActual PR (Swan 9): {actual_PR:.4f}")
print(f"Market Drop: {market_drop_val:.4f}")

### Experiment cell 63


In [ ]:
# INJECTED RESULT
injected_result = pd.DataFrame(index=stock_price.index)
S1 = (10000 * weight['weight'].values) / stock_price.iloc[0].values
injected_result["portfolio_value"] = stock_price.values @ S1
S2 = 10000 / float(OEX.iloc[0])
injected_result["oex"] = S2 * OEX.values
injected_result.to_csv(f"Injected_result_m{method_number}.csv")

# PR from injected result
temp = injected_result.loc[["2020-02-19", "2020-03-23"]]
Portfolio_DD = abs(temp['portfolio_value'].iloc[1] / temp['portfolio_value'].iloc[0] - 1)
DD = abs(temp['oex'].iloc[1] / temp['oex'].iloc[0] - 1)
PR = 1 - (Portfolio_DD / DD)
print(f"PR (from portfolio values): {PR:.4f}")

### Experiment cell 64


In [ ]:
# POST-ALLOCATION
allocation = pd.read_excel(DATA_DIR / 'allocation weight.xlsx', index_col='stock selected')

def post_allocation(dfMVOweight):
    dfMVOweight['equal'] = 1 / len(dfMVOweight)
    numofstocks = len(dfMVOweight)
    allocationweight = allocation.loc[numofstocks, :]
    dfMVOweight['proportion'] = np.nan
    dfMVOweight['adjusted proportion'] = np.nan
    dfMVOweight['direct'] = np.nan
    residual = 0
    for i in range(numofstocks):
        dfMVOweight.iloc[i, 4] = dfMVOweight.iloc[i, 1] / np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i, 4] + residual / (numofstocks - i)) > allocationweight.iloc[0]:
            dfMVOweight.iloc[i, 5] = allocationweight.iloc[0]
        else:
            dfMVOweight.iloc[i, 5] = (dfMVOweight.iloc[i, 4] + residual / (numofstocks - i))
        residual += dfMVOweight.iloc[i, 4] - dfMVOweight.iloc[i, 5]
    for i in range(int(allocationweight.iloc[1])):
        dfMVOweight.iloc[i, 6] = allocationweight.iloc[0]
    for i in range(int(allocationweight.iloc[1]), int(allocationweight.iloc[1] + allocationweight.iloc[3])):
        dfMVOweight.iloc[i, 6] = allocationweight.iloc[2]
    return dfMVOweight

dfMVOstock = deepcopy(weight)
dfMVOstock['PR_SR'] = np.nan
dfMVOstock['SR'] = np.nan
for i in range(len(weight)):
    stock_name = weight.index[i]
    if stock_name in preselected_stock.index:
        dfMVOstock.loc[stock_name, 'PR_SR'] = preselected_stock.loc[stock_name, 'PR_SR']
        dfMVOstock.loc[stock_name, 'SR'] = preselected_stock.loc[stock_name, 'SR']

dfMVOstock_sort = dfMVOstock.sort_values(by='PR_SR', ascending=False)
postallocationweight = post_allocation(dfMVOstock_sort)
print("Post-allocation weights:")
postallocationweight

### Experiment cell 65


In [ ]:
# OUTPUT FUNCTION & COMPUTE ALL WEIGHT SCHEMES
def compute_output(w, stock_price_all, OEX_series):
    w_clean = w.dropna()
    w_clean = w_clean[w_clean > 0]
    if len(w_clean) == 0:
        return None
    sp = stock_price_all[w_clean.index]
    result = pd.DataFrame(index=sp.index)
    S = (10000 * w_clean.values) / sp.iloc[0].values
    result["portfolio_value"] = sp.values @ S
    S2_local = 10000 / float(OEX_series.iloc[0])
    result["oex"] = S2_local * OEX_series.values
    return result

all_stocks_in_post = postallocationweight.index.tolist()
stock_price_all = get_data_yahoo(all_stocks_in_post, start=start_swan_date, end=end_swan_date)["Close"]
if isinstance(stock_price_all, pd.Series):
    stock_price_all = stock_price_all.to_frame(name=all_stocks_in_post[0])
stock_price_all = stock_price_all[all_stocks_in_post]

MVOweight = postallocationweight['weight']
equalweight = postallocationweight['equal']
proportionweight = postallocationweight['adjusted proportion']
directweight = postallocationweight['direct']

compute_output(MVOweight, stock_price_all, OEX).to_csv(f'no_post_allocation_result_m{method_number}.csv')
compute_output(equalweight, stock_price_all, OEX).to_csv(f'equal_weight_result_m{method_number}.csv')
compute_output(proportionweight, stock_price_all, OEX).to_csv(f'proportional_weight_result_m{method_number}.csv')
direct_out = compute_output(directweight, stock_price_all, OEX)
if direct_out is not None:
    direct_out.to_csv(f'direct_weight_result_m{method_number}.csv')
else:
    print("Warning: directweight has no valid stocks, using MVO fallback.")
    direct_out = compute_output(MVOweight, stock_price_all, OEX)
    direct_out.to_csv(f'direct_weight_result_m{method_number}.csv')
print("All weight scheme results saved.")

### Experiment cell 66


In [ ]:
# COMBINE RESULTS
mvo_result = pd.read_csv(f'no_post_allocation_result_m{method_number}.csv', index_col=0, parse_dates=True)
equal_result = pd.read_csv(f'equal_weight_result_m{method_number}.csv', index_col=0, parse_dates=True)
proportional_result = pd.read_csv(f'proportional_weight_result_m{method_number}.csv', index_col=0, parse_dates=True)
direct_result = pd.read_csv(f'direct_weight_result_m{method_number}.csv', index_col=0, parse_dates=True)

all_results = pd.DataFrame(index=injected_result.index)
all_results['MVO_portfolio'] = mvo_result['portfolio_value'].values
all_results['Equal_portfolio'] = equal_result['portfolio_value'].values
all_results['Proportional_portfolio'] = proportional_result['portfolio_value'].values
all_results['Direct_portfolio'] = direct_result['portfolio_value'].values
all_results['Injected_portfolio'] = injected_result['portfolio_value']
all_results['OEX'] = injected_result['oex']
all_results['Difference'] = all_results['Injected_portfolio'] - all_results['OEX']

all_results

### Experiment cell 67


In [ ]:
# PLOT
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(all_results.index, all_results['Injected_portfolio'], color='orange', linewidth=2, label='Injected Portfolio')
ax.plot(all_results.index, all_results['OEX'], color='grey', linewidth=2, label='OEX (S&P 100)')
ax.plot(all_results.index, all_results['Difference'], color='green', linewidth=2, label='Difference (Portfolio - OEX)')

swan_start = pd.Timestamp('2020-02-19')
swan_low = pd.Timestamp('2020-03-23')
ax.axvline(x=swan_start, color='black', linestyle='--', linewidth=1.2, label='Swan Start (2020-02-19)')
ax.axvline(x=swan_low, color='black', linestyle=':', linewidth=1.2, label='Swan Low (2020-03-23)')

ax.legend(loc='center right', fontsize=10, framealpha=0.9)
ax.set_title(
    f'Proj 27 Optimization-1: PR Maximization\n'
    f'Method: {method_name} | '
    f'Type E preselection of {len(weight)} stocks with k means = {N} in S&P 100\n'
    f'from swan 4-8 ----> 9 with MVO DB, MA = {MA_days}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'backtest_method{method_number}_plot.png', dpi=150)
plt.show()

### Experiment cell 68


In [ ]:
# PRINT ALL VALUES
print("=== Injected Portfolio Values ===")
for date, val in all_results['Injected_portfolio'].items():
    print(f"{date.strftime('%Y-%m-%d')}: {val:.4f}")

print("\n=== OEX Values ===")
for date, val in all_results['OEX'].items():
    print(f"{date.strftime('%Y-%m-%d')}: {val:.4f}")

print("\n=== Difference (Portfolio - OEX) ===")
for date, val in all_results['Difference'].items():
    print(f"{date.strftime('%Y-%m-%d')}: {val:.4f}")

### Experiment cell 69


In [ ]:
# SUMMARY
print(f"{'='*60}")
print(f"SUMMARY - Method {method_number}: {method_name}")
print(f"{'='*60}")
print(f"Actual PR (Swan 9):  {PR:.4f}")
print(f"Market Drop:         {market_drop_val:.4f}")
print(f"Portfolio Drop:      {Portfolio_DD:.4f}")
print(f"Stocks in portfolio: {len(weight)}")